# Hyperpatameter Tunning for Classification

In [ ]:
# Imports
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
from sklearn.model_selection import train_test_split , RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from scipy.stats import uniform , randint
from src.data_preprocessing import processed_dataset , get_feature_table

In [ ]:
# Loading preprocessed Dataset

customers = pd.read_csv(r"dataset\customers.csv")
transactions = pd.read_csv(r"dataset\transactions.csv")
products = pd.read_csv(r"dataset\products.csv")

df = processed_dataset(customers , transactions, products)
df

,quantity_sum,unit_price_mean,total_amount_sum,discount_applied_sum,discount_applied_mean,shipping_cost_sum,shipping_cost_mean,age_first,lifetime_value_max,transaction_id_count,...,payment_method_bank_transfer_sum,payment_method_credit_card_sum,payment_method_debit_card_sum,payment_method_google_pay_sum,payment_method_paypal_sum,status_cancelled_sum,status_completed_sum,status_pending_sum,status_refunded_sum,is_churned_first
0,32,34.675200,970.60,115,4.600000,144.08,5.763200,28,1595.27,25,...,4.0,7.0,3.0,3.0,3.0,2.0,18.0,2.0,3.0,0
1,22,32.725833,786.16,100,8.333333,42.33,3.527500,22,1160.61,12,...,0.0,5.0,3.0,0.0,3.0,3.0,5.0,4.0,0.0,0
2,39,58.933448,2112.49,195,6.724138,131.13,4.521724,30,3093.32,29,...,1.0,13.0,4.0,0.0,5.0,4.0,16.0,6.0,3.0,0
3,40,31.813462,1341.06,210,8.076923,118.08,4.541538,48,2131.08,26,...,0.0,9.0,5.0,1.0,8.0,2.0,15.0,3.0,6.0,1
4,4,46.135000,184.54,25,6.250000,19.31,4.827500,37,583.39,4,...,1.0,2.0,0.0,1.0,0.0,0.0,2.0,2.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9936,19,92.843333,1720.89,85,7.083333,37.70,3.141667,18,992.03,12,...,1.0,4.0,3.0,1.0,3.0,1.0,5.0,3.0,3.0,0
9937,10,39.595000,395.95,30,3.000000,51.22,5.122000,40,987.61,10,...,2.0,1.0,2.0,1.0,2.0,2.0,6.0,2.0,0.0,0
9938,3,21.713333,65.14,50,16.666667,24.28,8.093333,18,571.01,3,...,0.0,0.0,1.0,0.0,2.0,0.0,1.0,1.0,1.0,1
9939,13,26.944444,312.07,75,8.333333,64.75,7.194444,25,754.09,9,...,0.0,4.0,0.0,0.0,2.0,0.0,7.0,2.0,0.0,0


In [13]:
# 1st Random try

X = get_feature_table(df)
y = df["is_churned_first"]

X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

cat_cols = X.select_dtypes(include = ["object" , "string"]).columns
num_cols = X.select_dtypes(include = ["number"]).columns

preprocess = ColumnTransformer([("cat" , OneHotEncoder(handle_unknown="ignore") , cat_cols) ,
                                ("num" , StandardScaler() , num_cols)])

pipeline = Pipeline([
    ("preprocess" , preprocess),
    ("xgbc" , XGBClassifier(
        n_jobs = -1 ,
        random_state = 42 ,
        subsample = 0.8 ,
        colsample_bytree = 0.8
    ))
])


param_grid = {
    "xgbc__n_estimators" : randint(300 , 400) ,
    "xgbc__max_depth" :  randint(4,7),
    "xgbc__learning_rate" : uniform(0.01 , 0.09) ,
    "xgbc__gamma" : uniform(0.5 , 0.8) ,
    "xgbc__reg_lambda" : randint(2,7) ,
    "xgbc__reg_alpha" : uniform(0.01 , 0.02) ,
    "xgbc__min_child_weight" : [1,2] 
}

srch = RandomizedSearchCV(
    estimator = pipeline ,
    param_distributions = param_grid,
    n_iter = 20 ,
    cv = 5
)

srch.fit(X_train , y_train)

model = srch.best_estimator_

print(srch.best_params_)

{'xgbc__gamma': np.float64(1.0105869497056703), 'xgbc__learning_rate': np.float64(0.04066464017694031), 'xgbc__max_depth': 4, 'xgbc__min_child_weight': 1, 'xgbc__n_estimators': 315, 'xgbc__reg_alpha': np.float64(0.022929124916073976), 'xgbc__reg_lambda': 6}


# Considering {max_depth = 4} & {min_child_weight = 1} & {n_estimator = 355} as fixed baseline

In [16]:
# 1st Random try

X = get_feature_table(df)
y = df["is_churned_first"]

X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

cat_cols = X.select_dtypes(include = ["object" , "string"]).columns
num_cols = X.select_dtypes(include = ["number"]).columns

preprocess = ColumnTransformer([("cat" , OneHotEncoder(handle_unknown="ignore") , cat_cols) ,
                                ("num" , StandardScaler() , num_cols)])

pipeline = Pipeline([
    ("preprocess" , preprocess),
    ("xgbc" , XGBClassifier(
        n_jobs = -1 ,
        random_state = 42 ,
        subsample = 0.8 ,
        colsample_bytree = 0.8,
        max_depth = 4,
        min_child_weight = 1,
        n_estimators = 355
    ))
])


param_grid = {
    "xgbc__learning_rate" : uniform(0.02 , 0.06) ,
    "xgbc__gamma" : uniform(0.8 ,1.2) ,
    "xgbc__reg_lambda" : uniform(4,10) ,
    "xgbc__reg_alpha" : uniform(0.01 , 0.02) ,
}

srch = RandomizedSearchCV(
    estimator = pipeline ,
    param_distributions = param_grid,
    n_iter = 20 ,
    cv = 5
)

srch.fit(X_train , y_train)

model = srch.best_estimator_

print(srch.best_params_)

{'xgbc__gamma': np.float64(1.22294047843519), 'xgbc__learning_rate': np.float64(0.0219525914615242), 'xgbc__reg_alpha': np.float64(0.01067145599916558), 'xgbc__reg_lambda': np.float64(13.644393853513007)}


# Final Verdict
# ------------------------------------------
# max_depth = 4
# min_child_weight = 1
# gamma = 1.2222
# learning_rate = 0.0219
# alpha = 0.0106
# lambda = 14
# n_estimators = 355